# CCL Reasoning Stability Observatory

This notebook is the Course Correct Labs Reasoning Stability Observatory.

It loads results from three canonical empirical studies:

- **Mirror Loop** – measures whether iterative self-critique plateaus (rolling-3-step ΔI decay below threshold) and ΔI drift.
- **Recursive Confabulation** – measures how often fabricated claims persist across turns and how interventions change that.
- **Violation State** – measures safety-trigger contamination and how refusals bleed into unrelated queries.

Using these studies, the notebook builds a unified evaluation:

- Loads all three datasets (from their GitHub repos).
- Computes cross-study metrics per model.
- Generates a leaderboard and three-panel comparison figure.
- Exports a short markdown report of the results.

Run this notebook from top to bottom to get a single snapshot of "reasoning stability" across all models that Course Correct Labs has evaluated so far.

Echo Chamber Zero, an independent Course Correct Labs theoretical/systemic research
project, is **not** part of this canonical evaluation. Its previously implemented,
noncanonical Observatory integration is retained for provenance in the appendix at
the end of this notebook and does not run as part of the canonical workflow above.


## Setup & Installation

In [ ]:
# Setup: ensure we use the REPO version of course_correct_evals, not a stale pip wheel

import sys, os, subprocess

REPO_URL = "https://github.com/Course-Correct-Labs/course-correct-evals.git"
BRANCH = os.getenv("CCL_EVALS_BRANCH", "main")
REPO_DIR = "/content/course-correct-evals"

print(f"🔀 Using course-correct-evals branch: {BRANCH}")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🔧 Detected Google Colab environment")

    # 1) Uninstall any pip-installed version so it stops polluting imports
    try:
        print("🧹 Uninstalling pip package course-correct-evals (if present)...")
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "course-correct-evals"],
            check=False,
        )
    except Exception as e:
        print(f"⚠️ pip uninstall failed (ignoring): {e}")

    # 2) Clear any already-imported modules from this session
    for name in list(sys.modules.keys()):
        if name.startswith("course_correct_evals"):
            del sys.modules[name]
    print("🧽 Cleared course_correct_evals from sys.modules")

    # 3) Clone or update the repo
    if not os.path.exists(REPO_DIR):
        print(f"📦 Cloning repo from {REPO_URL} (branch {BRANCH})...")
        subprocess.run(
            ["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR],
            check=True,
        )
        print("✅ Repo cloned")
    else:
        print("✅ Repo already cloned, pulling latest...")
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=False)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

    # 4) Ensure repo is at the front of sys.path
    if REPO_DIR in sys.path:
        sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    print(f"✅ Added {REPO_DIR} to sys.path first")

else:
    # Local usage: assume user already has repo and editable install
    print("🏠 Not in Colab.")
    print("   For local use, run from inside the cloned repo or `pip install -e .`")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\n📚 Loading libraries...")

import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from course_correct_evals import (
    MirrorLoopImporter,
    ConfabulationImporter,
    ViolationStateImporter,
    EchoChamberImporter,
    CrossStudyAnalysis,
)

from course_correct_evals.analysis.viz import (
    plot_four_panel_comparison,
    plot_leaderboard,
    plot_mirror_loop_detail,
)

from course_correct_evals.reports import export_csv_results, export_pdf_report

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

# Plot settings
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")

warnings.filterwarnings("ignore")

import course_correct_evals, inspect
print("\n✅ Setup complete – ready to run!")
print("📍 course_correct_evals loaded from:", inspect.getfile(course_correct_evals))

## 1. Load All Studies

The Observatory will attempt to load data from the three canonical studies
(Mirror Loop, Recursive Confabulation, Violation State).
Studies without available data will be skipped gracefully.


In [ ]:
# Initialize Observatory
observatory = CrossStudyAnalysis()

# Load all available studies
loaded_studies = observatory.load_all_studies(
    fail_on_missing=False  # Continue even if some studies missing
)

print("\nLoading Summary:")
data_sources = observatory.get_data_source_summary()
for study_name, info in data_sources.items():
    status_icon = "[LOADED]" if info['loaded'] else "[UNAVAILABLE]"
    source = info['source']
    
    # Format source for readability
    if source.startswith('remote:'):
        source_desc = "GitHub (remote)"
    elif source.startswith('local:'):
        source_desc = f"Local file"
    elif source.startswith('explicit_path:'):
        source_desc = f"Explicit path"
    elif source.startswith('env:'):
        source_desc = f"Environment variable"
    else:
        source_desc = "Not loaded"
    
    print(f"  {status_icon} {study_name:20s} - {source_desc}")

In [ ]:
# Quick data sanity checks

print("\n============================================================")
print("DATA SANITY CHECKS")
print("============================================================")

# Basic counts per study
for name in ["mirror_loop", "confabulation", "violation_state"]:
    loaded = observatory._data_loaded.get(name, False)
    df = getattr(observatory, f"{name}_data", None) if loaded else None
    if loaded and df is not None:
        print(f"[{name}] rows: {len(df)}")
    else:
        print(f"[{name}] data not loaded or empty")

# Show leaderboard row count if already computed
try:
    lb = observatory.create_leaderboard()
    print(f"\nLeaderboard models: {len(lb)}")
except Exception as e:
    print("\nCould not compute leaderboard yet (this is not fatal):", e)

## 2. Cross-Study Leaderboard

Compare model performance across all four studies.

In [ ]:
# Export leaderboard to CSV for downstream analysis / dashboards

leaderboard = observatory.create_leaderboard()
csv_path = os.path.join(RESULTS_DIR, "observatory_leaderboard.csv")
leaderboard.to_csv(csv_path, index=False)

print(f"✅ Leaderboard exported to {csv_path}")

In [ ]:
# Generate leaderboard
leaderboard = observatory.create_leaderboard()

print("\nCross-Study Model Leaderboard:")
print("=" * 80)
display(leaderboard)

# Visualize leaderboard
if len(leaderboard) > 0:
    plot_leaderboard(leaderboard)

### Leaderboard Metrics Explanation

- **mirror_plateau_rate_grounded** / **mirror_plateau_rate_ungrounded**: manuscript-defined plateau rate (rolling-3-step ΔI mean below τ=0.05, PRIMARY threshold, per sequence then aggregated), by condition (no single collapse/pooled scalar; interpretation is study-dependent, not simply "lower is better")
- **confab_persist_rate_baseline** / **confab_persist_rate_fact_table** / **confab_persist_rate_belief_audit** / **confab_persist_rate_grounding_pilot**: released persistence-rate measurements for each model under that specific intervention arm (lower is better). These are measurements under distinct experimental conditions, not four interchangeable global model-quality scores or an independent ranking metric -- no averaging across arms is performed.


## 3. Three-Panel Comparison Figure

Publication-quality visualization comparing the three canonical studies.


In [ ]:
# Generate flagship figure
fig = plot_four_panel_comparison(
    observatory,
    figsize=(18, 6),
    save_path=os.path.join(RESULTS_DIR, "three_panel_comparison.png")
)

plt.show()


## 4. Mirror Loop Deep Dive

Manuscript-defined plateau analysis: per-sequence rolling-3-step ΔI average, first window whose mean falls below τ (τ=0.05 PRIMARY), reported at that window's last iteration; sequences with no qualifying window are "not plateaued" (never a fabricated iteration). Detection happens per sequence first, then results are aggregated by model × condition — never a threshold crossing on a pooled/averaged trajectory. A τ=0.02 sensitivity view is shown separately and never feeds the primary result. The grounding-rebound finding (grounded-condition pooled ΔI, iteration 2 vs. 4) is a distinct finding, not derived from plateau.

In [ ]:
if observatory._data_loaded['mirror_loop']:
    ml_analysis = observatory.analyze_mirror_loop()

    print("\nMirror Loop Study Results:")
    print("=" * 80)
    print(f"Total Sequences: {ml_analysis['total_sequences']}")
    print(f"Mean ΔI (released edit_change, overall): {ml_analysis['mean_delta_i_overall']:.3f}")

    print("\nPLATEAU RATE (manuscript-defined rolling-3-step statistic, tau=0.05 PRIMARY,")
    print("per-sequence detection then aggregated -- never a pooled-trajectory crossing):")
    plateau_rows = []
    for group_key, stats in ml_analysis['plateau']['group_summary'].items():
        label = ' / '.join(str(g) for g in group_key)
        iqr = stats['plateau_iteration_iqr']
        print(f"  {label}: {stats['n_plateaued']}/{stats['n_sequences']} plateaued "
              f"({stats['plateau_rate']:.1%})"
              + (f", median iter {stats['median_plateau_iteration']:.0f} (IQR {iqr[0]:.0f}-{iqr[1]:.0f})"
                 if stats['median_plateau_iteration'] is not None else ""))
        row = {gc: gv for gc, gv in zip(ml_analysis['plateau']['group_cols'], group_key)}
        row.update(stats)
        plateau_rows.append(row)
    display(pd.DataFrame(plateau_rows))

    print("\nSENSITIVITY VIEW (tau=0.02, SECONDARY -- does not feed the leaderboard and does")
    print("not alter the tau=0.05 primary result above):")
    sens_rows = []
    for group_key, stats in ml_analysis['plateau_sensitivity_tau_0_02']['group_summary'].items():
        label = ' / '.join(str(g) for g in group_key)
        print(f"  {label}: {stats['n_plateaued']}/{stats['n_sequences']} plateaued ({stats['plateau_rate']:.1%})")
        row = {gc: gv for gc, gv in zip(ml_analysis['plateau_sensitivity_tau_0_02']['group_cols'], group_key)}
        row.update(stats)
        sens_rows.append(row)
    display(pd.DataFrame(sens_rows))

    rebound = ml_analysis['grounding_rebound']
    if rebound is not None:
        print("\nGROUNDING REBOUND (manuscript-defined pooled ΔI comparison, grounded condition")
        print("only; a DISTINCT finding from plateau, not derived from or combined with it):")
        print(f"  iter {rebound['iteration_from']} -> {rebound['iteration_to']}: "
              f"{rebound['delta_i_from']:.6f} -> {rebound['delta_i_to']:.6f} "
              f"({rebound['pct_increase']:+.1f}%)")

    print("\nSample sequence detail (released ΔI, own plateau iteration marked if any):")
    sequence_ids = observatory.mirror_loop_data['sequence_id'].unique()
    if len(sequence_ids) > 0:
        plot_mirror_loop_detail(observatory, sequence_id=sequence_ids[0])
        plt.show()
else:
    print("Mirror Loop data not available")


## 5. Recursive Confabulation Deep Dive

Recursive Confabulation is not a single per-model scalar, and it involves two
DIFFERENT outcome variables that must not be substituted for one another:

- **View A -- manuscript pooled intervention comparison** (`persist_rate`): the
  N-weighted persistence rate for baseline / fact_table / belief_audit, pooled across
  the three models -- whether a fabrication *persists* after correction. This is the
  manuscript's primary tested comparison. `grounding_pilot` is deliberately excluded
  here, matching the source study's own pooled comparison.
- **View B -- grounding-confabulation heterogeneity** (`confab_rate`): whether the
  model *confabulates at all*, initially, under the `grounding_pilot` arm. The source
  study reports this as model-specific/heterogeneous ("Grounding reduced confabulation
  for GPT-4o mini only" -- README.md, RC_publication_pack.md). **This is a different
  outcome variable than View A** -- do not read View A's persistence numbers as
  evidence for the grounding-confabulation finding, or vice versa.

Per the manuscript (Section 4.3 -- not present in the audited recursive-confabulation
GitHub repository's artifacts; the repo contains no formal significance test for
grounding), GPT-4o Mini alone showed a statistically significant confabulation
reduction under grounding (p = 0.033). That significance claim is manuscript-sourced,
not independently reproducible from this repository's released data, and is not
re-derived here.

Grounding's effect on *persistence* specifically (as opposed to confabulation) is also
a real released measurement -- visible in the full model × arm table below and in the
`confab_persist_rate_grounding_pilot` leaderboard column -- but it is a separate
measurement, not the manuscript's grounding-confabulation finding, and is not promoted
to its own named canonical view here.


In [ ]:
if observatory._data_loaded['confabulation']:
    conf_analysis = observatory.analyze_confabulation()

    print("\nRecursive Confabulation Study Results:")
    print("=" * 80)
    print(f"Models: {', '.join(conf_analysis['models'])}")
    print(f"Intervention arms: {', '.join(conf_analysis['arms'])}")
    print(f"Total conversations: {conf_analysis['total_conversations']}")

    print("\nVIEW A -- Manuscript pooled intervention comparison (persist_rate)")
    print("(N-weighted across models; grounding_pilot excluded, matching the source study):")
    for arm, stats in conf_analysis['pooled_intervention_comparison'].items():
        print(f"  {arm}: {stats['persist_rate']:.2%} (N={stats['n']})")

    print("\nVIEW B -- Grounding-confabulation heterogeneity (confab_rate, model-specific, NOT pooled)")
    print("Source study finding: 'Grounding reduced confabulation for GPT-4o mini only'")
    print("(README.md / RC_publication_pack.md). Per the manuscript (Section 4.3, not")
    print("present in this repository's artifacts), GPT-4o Mini alone was statistically")
    print("significant (p=0.033) -- that significance figure is manuscript-sourced, not")
    print("re-derived here.")
    for model, stats in conf_analysis['grounding_confabulation_heterogeneity'].items():
        print(f"  {model}: {stats['confab_rate']:.2%} (N={stats['n']})")
    print("(Grounding's effect on PERSISTENCE specifically is a different, separate")
    print(" released measurement -- see the full model x arm table below, or the")
    print(" confab_persist_rate_grounding_pilot leaderboard column -- it is NOT this")
    print(" confabulation finding and must not be read as evidence for it.)")

    print("\nFull model x arm table -- all 12 released (model, arm) measurements")
    print("(both confab_rate and persist_rate shown together for transparency):")
    display(conf_analysis['model_arm_table'][['model', 'arm', 'n', 'confab_rate', 'persist_rate']])
else:
    print("Confabulation data not available")


## 6. Violation State Deep Dive

Canonical Violation State analysis uses the study's structured experimental
fields (condition, prompt_id, turn ordering, response_class) restricted to
the four canonical benign post-trigger image prompts — not generic text
classification.

Two provenance layers are shown, both derived from the same collapsed
per-(conversation, prompt) representation:

- **RAW structured outcomes**: 115 `policy_refusal`, 4 `image_success`, and
  1 terminal, never-retried `rate_limit` (contaminated, N=120); 40
  `image_success` / 0 refusals (control, N=40). Released response classes
  are shown as-is — the terminal rate-limit is never relabeled as a refusal.
- **PUBLISHED/HISTORICAL aggregate**: 116/120 = 96.67% (contaminated), 0/40
  = 0% (control) — the aggregation convention used by the study's final
  analysis (`analysis/run_analysis.py` in the violation-state repository),
  which counts that one terminal unresolved rate-limit within the
  refusal/failure tally. Both are shown so the released structured data and
  the study's own published convention stay distinguishable rather than
  being silently collapsed into one number.


In [ ]:
if observatory._data_loaded['violation_state']:
    vs_analysis = observatory.analyze_violation_state()
    structured = vs_analysis['structured']

    print("\nViolation State Study Results:")
    print("=" * 80)
    print(f"Total Conversations: {vs_analysis['total_conversations']}")
    print(f"Total Turns: {vs_analysis['total_turns']}")

    print("\nRAW STRUCTURED OUTCOMES (released data, as observed):")
    for cond, data in structured['raw_structured_outcomes'].items():
        print(f"  {cond} (N={data['n']}): {data['counts']}")

    print("\nPUBLISHED/HISTORICAL AGGREGATE (historical rate-limit-as-refusal convention):")
    for cond, data in structured['published_aggregate'].items():
        print(f"  {cond}: {data['refused']}/{data['n']} refused ({data['refusal_rate']:.2%})")

    print("\nNote: the terminal, never-retried rate_limit in the contaminated condition is")
    print("preserved as 'rate_limit' in the raw layer above, and is only folded into the")
    print("'refused' tally in the published/historical aggregate — it is not presented as")
    print("an observed policy refusal.")

    print("\nCollapsed per-(conversation, prompt) representation (head):")
    display(structured['collapsed'].head(10))
else:
    print("Violation State data not available")


## 7. Optional: Live Demo

**⚠️ WARNING: This cell costs money and requires API keys!**

To enable:
1. Set `RUN_LIVE_DEMO = True` below
2. Provide your API key
3. Run the cell

Estimated cost: $0.01-0.05 for 10 iterations


In [ ]:
# DISABLED BY DEFAULT
RUN_LIVE_DEMO = False

if RUN_LIVE_DEMO:
    print("WARNING: Running live demo - this will cost money!")
    
    from course_correct_evals.runners.mirror_loop_runner import (
        run_mirror_loop_demo,
        analyze_live_demo,
    )
    
    import course_correct_evals.runners.mirror_loop_runner as runner_module
    runner_module.RUN_LIVE_DEMO = True
    
    demo_results = run_mirror_loop_demo(
        prompt="Explain the concept of recursion in programming.",
        model="gpt-3.5-turbo",
        max_iterations=10,
        api_key=None
    )
    
    demo_analysis = analyze_live_demo(demo_results)
    
    print("\nLive Demo Results:")
    print("=" * 80)
    print(f"Sequence Length: {demo_analysis['length']}")
    print(f"Collapse Detected: {demo_analysis['collapse_detected']}")
    if demo_analysis['collapse_detected']:
        print(f"Collapse at Iteration: {demo_analysis['collapse_iteration']}")
    print(f"Mean ΔI: {demo_analysis['delta_i_edit_mean']:.3f}")
    print(f"Mean N-gram Novelty: {demo_analysis['ngram_novelty_mean']:.3f}")
    
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(demo_analysis['delta_i_edit']) + 1),
             demo_analysis['delta_i_edit'],
             'o-', linewidth=2, markersize=6)
    plt.axhline(y=demo_analysis['collapse_threshold'],
                color='red', linestyle='--', alpha=0.5,
                label=f"Threshold ({demo_analysis['collapse_threshold']})")
    plt.xlabel('Iteration')
    plt.ylabel('ΔI (Edit Distance)')
    plt.title('Live Demo: Information Change Over Iterations')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Live demo disabled. Set RUN_LIVE_DEMO = True to enable.")
    print("Note: This will cost money and require API keys.")

## 8. Export Results

Export all analysis results to CSV and generate a report.


In [ ]:
# Export CSV results
print("Exporting results to CSV...\n")
exported_files = export_csv_results(observatory, output_dir=RESULTS_DIR)

print("\nExported Files:")
for result_type, file_path in exported_files.items():
    print(f"  {result_type}: {file_path}")

In [ ]:
# Generate report
print("Generating report...\n")
report_path = export_pdf_report(
    observatory,
    output_path=os.path.join(RESULTS_DIR, "ccl_observatory_report.pdf")
)

print(f"\nReport generated: {report_path}")

## Summary

This notebook has provided a comprehensive analysis of the CCL Reasoning Stability Observatory.

### Key Findings

The Observatory synthesizes data across three canonical empirical studies to identify:

1. **Plateau** - Whether models' iterative self-critique plateaus (rolling-3-step ΔI decay below τ)
2. **Fabrication Persistence** - Models prone to maintaining false information
3. **State Contamination** - Models whose refusal behavior leaks across contexts

Echo Chamber Zero results, where previously integrated, are retained as noncanonical
provenance in the Appendix below and are not part of these canonical findings.

### Next Steps

1. Review exported CSV results in `results/` folder (or download from Colab)
2. Examine the three-panel comparison figure
3. Use the leaderboard to compare model stability
4. Optionally: Run live demos with your own prompts

---

**Course Correct Labs**
*Reasoning Stability Observatory v0.1.0*


## Appendix — Echo Chamber Zero (Decoupled / Non-Canonical)

Echo Chamber Zero is an independent theoretical/systemic Course Correct Labs research
project — a percolation-based model of synthetic epistemic drift on a provenance
graph. It is **not** part of the canonical Reasoning Stability Observatory evaluation
set.

**The canonical Observatory consists of:**
1. Mirror Loop
2. Recursive Confabulation
3. Violation State

This appendix is retained for provenance and historical transparency, documenting the
previously implemented Observatory integration. It does **not** execute automatically
as part of the canonical top-to-bottom notebook workflow above — running it requires
explicitly setting `RUN_ECHO_CHAMBER_APPENDIX = True` below, which itself triggers an
explicit opt-in data load (`include_echo_chamber=True`). Setting `include_echo_chamber`
to `True` on its own, elsewhere, does not execute this appendix.

Its presence here is **not** validation of the Echo Chamber Zero simulation's currently
open reproducibility question (a live full-scale reproduction did not match the
manuscript's reported thresholds under the frozen methodology — see the separate
Echo Chamber Zero repair track). Its outputs are **not** part of any canonical
dashboard, leaderboard, or model-comparison conclusion above.

**Corrected terminology** (the original integration used incorrect names):
- **GR = Groundedness Ratio** (not "Group Radicalization")
- **SRI = Synthetic Recurrence Index** (not "Self-Reinforcement Index")
- **RE = Referential Entropy** (not "Reasoning Entropy")

This is a percolation simulation on a provenance network, not a multi-agent
belief-radicalization study.


In [ ]:
# DISABLED BY DEFAULT — Echo Chamber Zero is noncanonical/opt-in and does not run
# as part of the canonical top-to-bottom Observatory workflow. Both the explicit data
# opt-in (include_echo_chamber=True) and this flag must be set for anything below to
# execute; setting include_echo_chamber=True elsewhere does NOT by itself run this cell.
RUN_ECHO_CHAMBER_APPENDIX = False

if RUN_ECHO_CHAMBER_APPENDIX:
    print("Loading Echo Chamber Zero data (noncanonical, opt-in)...")
    observatory.load_all_studies(fail_on_missing=False, include_echo_chamber=True)

    if observatory._data_loaded['echo_chamber']:
        echo_analysis = observatory.analyze_echo_chamber()

        print("\nEcho Chamber Zero Study Results (noncanonical):")
        print("=" * 80)
        print(f"Total Simulations: {echo_analysis['total_simulations']}")
        print(f"Total Steps: {echo_analysis['total_steps']}")

        echo_stats = echo_analysis['echo_statistics']
        print("\nEcho Chamber Zero Metrics (GR = Groundedness Ratio, SRI = Synthetic Recurrence Index, RE = Referential Entropy):")
        for metric_name, metric_stats in echo_stats['metrics'].items():
            print(f"\n{metric_name}:")
            print(f"  Mean: {metric_stats['mean']:.3f}")
            print(f"  Std: {metric_stats['std']:.3f}")
            print(f"  Range: [{metric_stats['min']:.3f}, {metric_stats['max']:.3f}]")
            if 'trend_mean' in metric_stats:
                print(f"  Mean Trend: {metric_stats['trend_mean']:.3f}")
                print(f"  Increasing Trajectories: {metric_stats['increasing_count']}")
                print(f"  Decreasing Trajectories: {metric_stats['decreasing_count']}")

        print("\nConvergence Statistics:")
        conv_stats = echo_analysis['convergence_statistics']
        for key, value in conv_stats.items():
            print(f"  {key}: {value}")

        print("\nSample Trajectories:")
        display(echo_analysis['trajectories'].head(10))

        print("\nThreshold Crossings:")
        for metric, crossings in echo_analysis['threshold_crossings'].items():
            if len(crossings) > 0:
                print(f"\n{metric}:")
                display(crossings.head())
    else:
        print("Echo Chamber Zero data not available")
else:
    print("Echo Chamber Zero appendix disabled. Set RUN_ECHO_CHAMBER_APPENDIX = True to enable.")
    print("Note: this is noncanonical/opt-in — see appendix markdown above.")
